In [2]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have

'''

import sys
from datetime import datetime
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running Date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# file_path = r"\sample_data\test-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted:", os.path.ismount('/content/drive'))

file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+file_path+" -", os.path.isfile(file_path))


# Read the Parquet file 
# with specific columns and filter for English language reviews
# For better understanding I have filtered the data for few columns English language reviews only. You can remove the filter if you want to read all the data.

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                      filters=[[('language', '==', 'en')]])

#Print the first few rows of the DataFrame to see how the data looks like.
df = table.to_pandas()
df.info()
print(df.head(4))
#print(df.shape)
print("Count of each star rating:"+str(df["stars"].value_counts().to_dict()))

'''DATA CLEANING AND LABEL PREPARATION
Here we will convert the "stars" column to numeric and create a new "sentiment" column based on the star ratings.
We wil keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
We might also need to drop and empty or null values in the review_body"stars" column, as they cannot be used for sentiment analysis. Since Data is clean commenting it out. If you want to use it, you can uncomment the code below.
'''

# df = df[["review_body", "stars"]].dropna().copy()

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
# df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)

# # Remove neutral examples for a simpler binary sentiment task
# df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

print(df[["stars", "sentiment"]].head(6))
print(df["stars"].value_counts())
print(df["sentiment"].value_counts())

Mounted at /content/drive
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_id    460 non-null    object
 1   review_body  460 non-null    object
 2   language     460 non-null    object
 3   stars        460 non-null    int32 
dtypes: int32(1), object(3)
memory usage: 12.7+ KB
    review_id                                        review_body language  \
0  en_0254382  Well I haven't stunned anyone...yet. So I can ...       en   
1  en_0961337  I loved the first book so much had to read the...       en   
2  en_0828526  This wasn't my favorite bag, but it wasn't my ...       en   
3  en_0910562  Good storyline but the characterizations seeme...       en   

   stars  
0      2  
1      1  
2      3  
3      3  
Count of each star rating:{1

In [3]:
import warnings
import datasets
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# --- SUPPRESS ALL EXCESS LOGS & PROGRESS BARS ---
warnings.filterwarnings("ignore")
datasets.disable_progress_bar()
from transformers import logging as transformers_logging

transformers_logging.set_verbosity_error()

!pip install -q optuna
import optuna  # Required for hyperparameter search

# 1. Dataset Setup (Assumes 'df' with 'review_body' and 'sentiment' columns is defined)
# df = pd.DataFrame(data)

# 2. Map string labels to integers (Required for Transformer models)
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
inv_label_mapping = {0: "negative", 1: "neutral", 2: "positive"}
df["label"] = df["sentiment"].map(label_mapping)

# 3. Train/Test Split (Stratified)
train_df, test_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)

# Convert Pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# 4. Tokenization using RoBERTa Tokenizer
# model_name = "answerdotai/ModernBERT-base"
# model_name = "answerdotai/ModernBERT-base"
model_name = "google/electra-base-discriminator"
# model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
  return tokenizer(examples["review_body"], truncation=True, max_length=128)


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print(f"Number of training samples: {len(tokenized_train)}")
print(f"Number of test samples: {len(tokenized_test)}")
print("Sample tokenized features:", tokenized_train[0]["input_ids"])


# 5. Model Initialization Function (Required for Optuna hyperparameter search)
def model_init(trial):
  return AutoModelForSequenceClassification.from_pretrained(
      model_name, num_labels=3, id2label=inv_label_mapping, label2id=label_mapping
  )


# 6. Define Evaluation Metric Function
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  preds = np.argmax(predictions, axis=1)
  accuracy = np.mean(preds == labels)
  return {"accuracy": accuracy}


# 7. Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./roberta_sentiment_results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,  # Parallelize computation
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_strategy="no",  # Suppress excess logging
    disable_tqdm=True,  # Suppress training progress bars
    report_to="none",
    fp16=True,  # 2x speedup via mixed precision
    dataloader_num_workers=2,  # Prevent CPU data bottlenecks
    per_device_eval_batch_size=32,  # Speeds up evaluation steps
)

# 8. Initialize Trainer with model_init
trainer = Trainer(
    model_init=model_init,  # Pass model_init instead of a single model instance
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


# 9. Define Hyperparameter Search Space for Optuna
def hp_space(trial):
  return {
      "learning_rate": trial.suggest_float(
          "learning_rate", 1e-5, 5e-5, log=True
      ),
      "per_device_train_batch_size": trial.suggest_categorical(
          "per_device_train_batch_size", [8, 16]
      ),
      "num_train_epochs": trial.suggest_int(
          "num_train_epochs", 2, 4
      ),  # Tests between 2 to 4 epochs
      "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
  }


# 10. Run Hyperparameter Tuning
print("Starting Optuna Hyperparameter Search with RoBERTa...")
best_run = trainer.hyperparameter_search(
    direction="maximize",  # We want to maximize accuracy
    backend="optuna",
    hp_space=hp_space,
    n_trials=3,  # Number of different hyperparameter combinations to test
)

print("\nBest Hyperparameters Found:")
print(best_run.hyperparameters)

# 11. Apply Best Hyperparameters & Finalize Training
print("\nApplying Best Hyperparameters for Final Training...")
for key, value in best_run.hyperparameters.items():
  setattr(trainer.args, key, value)

trainer.train()

# 12. Evaluate on the Test Set
print("\nEvaluating Best Model on Test Data:")
eval_results = trainer.evaluate(tokenized_test)
print(eval_results)

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Number of training samples: 322
Number of test samples: 138
Sample tokenized features: [101, 1996, 2203, 1997, 1996, 2482, 4770, 16142, 6669, 1999, 2087, 6081, 11015, 2013, 2300, 2030, 14904, 1012, 1996, 5331, 2003, 3811, 12379, 2437, 2009, 3733, 2000, 2597, 1998, 2224, 1012, 1045, 4033, 1005, 1056, 4149, 2023, 2004, 1037, 5592, 2664, 1010, 2021, 2933, 2000, 1012, 2036, 1010, 2026, 2564, 4122, 2000, 3288, 2023, 2006, 1037, 13039, 4440, 2061, 2016, 2064, 2224, 2009, 2096, 13039, 2006, 1996, 4446, 1999, 1996, 2067, 2406, 1012, 102]


...

{'eval_loss': '0.969', 'eval_accuracy': '0.6957', 'eval_runtime': '0.24', 'eval_samples_per_second': '575.1', 'eval_steps_per_second': '20.84', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '0.9204', 'eval_accuracy': '0.6957', 'eval_runtime': '0.235', 'eval_samples_per_second': '587.2', 'eval_steps_per_second': '21.28', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '12.03', 'train_samples_per_second': '80.3', 'train_steps_per_second': '5.237', 'train_loss': '0.9824', 'epoch': '3'}

Evaluating Best Model on Test Data:
{'eval_loss': '0.9204', 'eval_accuracy': '0.6957', 'eval_runtime': '0.2488', 'eval_samples_per_second': '554.7', 'eval_steps_per_second': '20.1', 'epoch': '3'}
{'eval_loss': 0.9204491376876831, 'eval_accuracy': 0.6956521739130435, 'eval_runtime': 0.2488, 'eval_samples_per_second': 554.693, 'eval_steps_per_second': 20.098, 'epoch': 3.0}
